# Kaggle Image → Video — LTX-Video 2B Distilled

This notebook **actually runs the model inside Kaggle** and writes a real `.mp4` to `/kaggle/working/outputs`.

Before running:
1. Kaggle → **Settings → Accelerator → GPU T4 x2**
2. Turn **Internet ON**
3. Run cells top-to-bottom.

Why LTX 2B here: it is much smaller and more realistic for Kaggle's T4 memory/RAM than the 34+ GB HunyuanVideo-1.5 Diffusers checkpoint.

The first model download can take a while. Later generations in the same session are much faster because the weights stay cached.

In [ ]:
# 1) Check the Kaggle GPU
import os, sys, torch

assert torch.cuda.is_available(), "No CUDA GPU found. In Kaggle Settings, set Accelerator = GPU T4 x2."

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} | {props.total_memory/1024**3:.1f} GB")

# Use one T4. LTX 2B fits on one with CPU offload; the second T4 is not required.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"


In [ ]:
# 2) Install official LTX-Video code.
# Put Hugging Face weights in Kaggle's temporary scratch space instead of /kaggle/working.
import os, subprocess, sys
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/tmp/hf"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/kaggle/tmp/hf/hub"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/tmp/hf/transformers"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

repo = Path("/kaggle/working/LTX-Video")

if not repo.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/Lightricks/LTX-Video.git",
        str(repo)
    ])

subprocess.check_call(
    f'{sys.executable} -m pip install -q -e ".[inference]"',
    cwd=str(repo),
    shell=True
)

print("Installed LTX-Video.")


In [ ]:
# 3) T4 compatibility patch + Kaggle-friendly config
#
# LTX's official 2B config uses BF16. Tesla T4 is a pre-Ampere GPU, so this notebook
# switches the inference path to FP16. We also disable the optional prompt-enhancer
# models so Kaggle does NOT download an extra multi-billion-parameter LLM.

from pathlib import Path
import re

repo = Path("/kaggle/working/LTX-Video")
src_cfg = repo / "configs/ltxv-2b-0.9.8-distilled.yaml"
t4_cfg = repo / "configs/ltxv-2b-0.9.8-distilled-t4.yaml"

cfg = src_cfg.read_text()
cfg = cfg.replace('precision: "bfloat16"', 'precision: "float16"')
cfg = re.sub(
    r'prompt_enhancement_words_threshold:\s*\d+',
    'prompt_enhancement_words_threshold: 0',
    cfg
)
t4_cfg.write_text(cfg)

inference_py = repo / "ltx_video/inference.py"
code = inference_py.read_text()

# Add an FP16 transformer branch if the upstream file does not already have one.
if 'precision == "float16"' not in code:
    pattern = (
        r'(elif precision == "bfloat16":\n'
        r'\s+return Transformer3DModel\.from_pretrained\(ckpt_path\)'
        r'\.to\(torch\.bfloat16\))'
    )
    replacement = (
        r'\1\n'
        r'    elif precision == "float16":\n'
        r'        return Transformer3DModel.from_pretrained(ckpt_path).to(torch.float16)'
    )
    code, n = re.subn(pattern, replacement, code)
    if n == 0:
        raise RuntimeError("Upstream LTX inference code changed; FP16 patch location was not found.")

# Keep VAE + text encoder on a dtype T4 supports well.
code = code.replace("vae = vae.to(torch.bfloat16)", "vae = vae.to(torch.float16)")
code = code.replace("text_encoder = text_encoder.to(torch.bfloat16)", "text_encoder = text_encoder.to(torch.float16)")
inference_py.write_text(code)

print("T4 config:", t4_cfg)
print("FP16 patch ready.")


## Upload your starting image

Run the next cell, tap **Upload**, choose an image from your phone/computer, then run the cell after it.

If Kaggle's widget does not appear, use Kaggle's **Add Input** button instead and manually set `IMAGE_PATH` to that image.

In [ ]:
# 4) Upload image
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload image"
)
display(uploader)


In [ ]:
# 5) Save uploaded image to /kaggle/working/input_image.png
from pathlib import Path
from io import BytesIO
from PIL import Image, ImageOps
from IPython.display import display

IMAGE_PATH = "/kaggle/working/input_image.png"

if uploader.value:
    value = uploader.value

    # ipywidgets 8 -> tuple/list; ipywidgets 7 -> dict
    if isinstance(value, dict):
        item = next(iter(value.values()))
    else:
        item = value[0]

    raw = bytes(item["content"])
    img = Image.open(BytesIO(raw))
    img = ImageOps.exif_transpose(img).convert("RGB")
    img.save(IMAGE_PATH)
else:
    raise RuntimeError(
        "No image uploaded. Run the upload cell first, or manually set IMAGE_PATH to an image under /kaggle/input."
    )

print("Saved:", IMAGE_PATH)
display(Image.open(IMAGE_PATH))


## Describe the motion

Describe **what changes after the first frame**, not the contents of the still image alone. LTX generally responds better to concrete motion: facial expression, body movement, hair/clothing movement, camera motion, and lighting.

In [ ]:
# 6) Your motion prompt + settings

PROMPT = """
The person in the reference image stays recognizable and naturally consistent.
They smile warmly, blink, subtly shift their posture, and give a small playful wave toward the camera.
Their hair moves slightly as they move. Natural realistic body motion and facial motion.
The camera makes a very gentle handheld push-in. Keep the same clothing, room, lighting, face, and identity.
Photorealistic candid video, coherent anatomy, smooth continuous motion, no scene change.
""".strip()

NEGATIVE_PROMPT = (
    "worst quality, blurry, jittery, inconsistent motion, distorted face, "
    "deformed hands, extra fingers, duplicate person, morphing identity, "
    "warped body, text, logo, watermark, scene cut"
)

SEED = 42
FPS = 24

# 65 frames = about 2.7 seconds at 24 fps.
# After your first successful test, try 97 or 121 frames.
NUM_FRAMES = 65

# Automatically choose a T4-friendly portrait / landscape size.
from PIL import Image
with Image.open(IMAGE_PATH) as im:
    w, h = im.size

if h > w:
    HEIGHT, WIDTH = 640, 384
elif w > h:
    HEIGHT, WIDTH = 384, 640
else:
    HEIGHT, WIDTH = 512, 512

print("Resolution:", WIDTH, "x", HEIGHT)
print("Frames:", NUM_FRAMES, "| duration:", round(NUM_FRAMES / FPS, 2), "sec")
print("Prompt:", PROMPT)


In [ ]:
# 7) GENERATE THE VIDEO
# The first run downloads the ~6.3 GB LTX 2B checkpoint + its small supporting models.
# This is local Kaggle inference — no paid API.

import sys, os, gc, torch
from pathlib import Path

sys.path.insert(0, "/kaggle/working/LTX-Video")
os.chdir("/kaggle/working/LTX-Video")

from ltx_video.inference import infer, InferenceConfig

OUTPUT_DIR = "/kaggle/working/outputs"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

gc.collect()
torch.cuda.empty_cache()

config = InferenceConfig(
    pipeline_config="/kaggle/working/LTX-Video/configs/ltxv-2b-0.9.8-distilled-t4.yaml",
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    conditioning_media_paths=[IMAGE_PATH],
    conditioning_strengths=[1.0],
    conditioning_start_frames=[0],
    height=HEIGHT,
    width=WIDTH,
    num_frames=NUM_FRAMES,
    frame_rate=FPS,
    seed=SEED,
    offload_to_cpu=True,
    output_path=OUTPUT_DIR,
)

infer(config)
print("Generation finished.")


In [ ]:
# 8) Show the newest generated MP4 + give you the exact output path
from pathlib import Path
from IPython.display import Video, display

videos = sorted(
    Path("/kaggle/working/outputs").glob("*.mp4"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not videos:
    raise FileNotFoundError("No MP4 was found in /kaggle/working/outputs")

OUTPUT_VIDEO = str(videos[0])
print("OUTPUT VIDEO:", OUTPUT_VIDEO)
display(Video(OUTPUT_VIDEO, embed=True))


## Better-quality second run

Once the 65-frame test works:

- Change `NUM_FRAMES = 97` or `121`
- For portrait, try `HEIGHT, WIDTH = 768, 448`
- Keep dimensions divisible by 32
- Rewrite the prompt with more specific chronological movement
- Change `SEED` to explore different motion

If you hit CUDA OOM, go back to `640 × 384` and `65` frames first.